# Google 2019 Cluster - Temporal Diagnostic (why it is NOT used for forecasting)

**Module:** temporal diagnostic &nbsp;|&nbsp; **Dataset:** Google 2019 Cluster

**Input file:** `borg_traces_data.csv`

This is **not** a forecasting experiment. It tests whether the public extract has any temporal structure a sequence model could learn. It does not.

Steps: build the hourly cell-level aggregate (mean **and** sum) -> interpolate missing buckets -> autocorrelation (ACF) at several lags -> white-noise noise band (1.96/sqrt(n)) -> compare with an Alibaba cell-level series.

**Result -> ** `results/temporal_diagnostic.json`

**Conclusion:** Google ACF is inside the noise band at every lag (white noise); Alibaba ACF is 0.89 at lag 1 and 0.55 at lag 24 (clear diurnal structure). Therefore the final LSTM/BiLSTM/RNN forecasting module uses **Alibaba 2017 only**; Google 2019 Cluster is used for cross-sectional regression only.

In [ ]:
# Step 1: Import libraries
import ast, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BUCKET_US = 3_600_000_000          # 1 hour, in microseconds
LAGS = [1, 2, 3, 6, 12, 24, 48]
RESULTS_DIR = r"../../results"
BORG_CSV = r"../../data/google_2019/borg_traces_data.csv"
ALIBABA_DIR = r"../../data/alibaba_2017"

# Step 2: Load the Google 2019 Cluster extract (only the 2 columns we need)
g = pd.read_csv(BORG_CSV, low_memory=False, usecols=["start_time", "average_usage"])
st = pd.to_numeric(g["start_time"], errors="coerce")
keep = st.between(1e6, 3e12)
g = g[keep].copy()
g["bucket"] = (st[keep] // BUCKET_US).astype("int64")   # hour index

# Step 3: parse average_usage -> cpus / memory
def dget(s, k):
    try:
        return ast.literal_eval(s).get(k, np.nan)
    except Exception:
        return np.nan

# Step 4-5: hourly cell-level aggregate  (MEAN and SUM)  + missing-bucket interpolation
def cell_series(resource, agg):
    v = g["average_usage"].map(lambda s: dget(s, resource))
    d = g.assign(val=v).dropna(subset=["val"])
    d = d[d["val"].between(0, 5)]
    s = d.groupby("bucket")["val"].mean() if agg == "mean" else d.groupby("bucket")["val"].sum()
    full = pd.RangeIndex(s.index.min(), s.index.max() + 1)          # regular hourly grid
    return s.reindex(full).interpolate("linear").bfill().ffill().to_numpy(float)

g_cpu_mean = cell_series("cpus", "mean")
g_cpu_sum  = cell_series("cpus", "sum")
g_mem_mean = cell_series("memory", "mean")
print("google hourly series length:", len(g_cpu_mean))

# Step 6: ACF  +  Step 7: noise band
def acf(x, lags=LAGS):
    x = (np.asarray(x, float) - np.mean(x)) / (np.std(x) + 1e-12)
    return {int(k): float(np.corrcoef(x[:-k], x[k:])[0, 1]) for k in lags}

def noise_band(n):
    return float(1.96 / np.sqrt(n))

# Step 8: comparison series -- Alibaba cell-level (mean of server_usage across machines per ts)
su = pd.read_csv(os.path.join(ALIBABA_DIR, "server_usage.csv"), header=None,
                 names=["ts", "machine_id", "cpu_util", "mem_util", "disk_util",
                        "load1", "load5", "load15"])
a_cpu = su.groupby("ts")["cpu_util"].mean().sort_index().to_numpy(float) / 100.0
a_mem = su.groupby("ts")["mem_util"].mean().sort_index().to_numpy(float) / 100.0

diag = {
    "google_cpus_mean": {"n": len(g_cpu_mean), "noise_band": round(noise_band(len(g_cpu_mean)), 4), "acf": acf(g_cpu_mean)},
    "google_cpus_sum":  {"n": len(g_cpu_sum),  "noise_band": round(noise_band(len(g_cpu_sum)), 4),  "acf": acf(g_cpu_sum)},
    "google_memory_mean": {"n": len(g_mem_mean), "noise_band": round(noise_band(len(g_mem_mean)), 4), "acf": acf(g_mem_mean)},
    "alibaba_cpu_util_cell": {"n": len(a_cpu), "noise_band": round(noise_band(len(a_cpu)), 4), "acf": acf(a_cpu, [1,2,3,6,12,24])},
    "alibaba_mem_util_cell": {"n": len(a_mem), "noise_band": round(noise_band(len(a_mem)), 4), "acf": acf(a_mem, [1,2,3,6,12,24])},
}

# Step 9: the resulting diagnostic
print(json.dumps(diag, indent=2))
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, "temporal_diagnostic.json"), "w") as fh:
    json.dump(diag, fh, indent=2)

# plot: Google (noise) vs Alibaba (structure)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(g_cpu_mean); ax[0].set_title("Google 2019 Cluster - hourly cell CPU (mean)  -> white noise")
ax[1].plot(a_cpu);      ax[1].set_title("Alibaba 2017 - cell CPU utilisation  -> diurnal structure")
plt.tight_layout(); plt.show()

gb, ab = diag["google_cpus_mean"], diag["alibaba_cpu_util_cell"]
lags_bar = [1, 2, 3, 6, 12, 24]
plt.figure(figsize=(9, 4))
plt.bar([l - 0.35 for l in lags_bar], [gb["acf"][l] for l in lags_bar], width=0.7, label="Google 2019 Cluster")
plt.bar([l + 0.35 for l in lags_bar], [ab["acf"][l] for l in lags_bar], width=0.7, label="Alibaba 2017")
plt.axhline( gb["noise_band"], ls="--", c="grey"); plt.axhline(-gb["noise_band"], ls="--", c="grey")
plt.xlabel("lag (hours)"); plt.ylabel("autocorrelation"); plt.legend()
plt.title("ACF: Google extract is inside the noise band at every lag"); plt.tight_layout(); plt.show()

# Step 10: reasoning / decision
print("\nDECISION")
print(f"  Google cell series (n={gb['n']}, band +-{gb['noise_band']:.3f}): "
      f"ACF lag1={gb['acf'][1]:+.3f}, lag24={gb['acf'][24]:+.3f}  -> statistically WHITE NOISE")
print(f"  Alibaba cell series (n={ab['n']}, band +-{ab['noise_band']:.3f}): "
      f"ACF lag1={ab['acf'][1]:+.3f}, lag24={ab['acf'][24]:+.3f}  -> clear diurnal STRUCTURE")
print("  => The public borg_traces_data.csv extract is a random cross-sectional sample and")
print("     cannot support sequence modelling. LSTM/BiLSTM/RNN would (and do) score R2 ~ 0 on it.")
print("  => FINAL DESIGN: Google 2019 Cluster -> cross-sectional regression only;")
print("     time-series forecasting (LSTM/BiLSTM/RNN) is reported on Alibaba 2017 only.")